# FDP6 Capstone — Pipeline End-to-End (MSFT)
**Tujuan notebook ini:** merangkai seluruh track menjadi satu pipeline — fetch → parse → clean → ratios — untuk Microsoft, menulis `dataset_kanonis.csv`, lalu menjalankan self-check ala C3 (konsistensi internal). Ini Bagian A dari capstone FDP6.

> Dijalankan: 4 Nov 2026. Bagian B (rekonsiliasi vs benchmark IDX) memakai `reconcile.py` + `benchmark.csv` di folder `fdp-tooling/`.

In [1]:
# Tujuan cell ini: fetch companyfacts MSFT (tahap 1: Fetch)
import json, csv
import urllib.request
from datetime import date
from decimal import Decimal

HEADERS = {"User-Agent": "Septian FDP-Learn septian@example.com"}
req = urllib.request.Request("https://data.sec.gov/api/xbrl/companyfacts/CIK0000789019.json", headers=HEADERS)
facts = json.load(urllib.request.urlopen(req, timeout=60))
print("fetch OK:", facts["entityName"])

fetch OK: MICROSOFT CORP


## Penjelasan (Indonesia)
Tahap **Fetch** (FDP2): satu request mengambil seluruh fakta XBRL MSFT. `User-Agent` wajib — tanpanya SEC mengembalikan 403. Di Colab, cell ini berjalan apa adanya; di browser biasa tidak bisa (CORS), itulah kenapa fetch tinggal di notebook.

In [1]:
# Tujuan cell ini: parse JSON mentah menjadi tabel fakta kanonis (tahap 2: Parse)
CONCEPTS = ["NetIncomeLoss", "Assets", "Liabilities", "StockholdersEquity"]

def is_annual(f):
    if f.get("form") != "10-K" or f.get("fp") != "FY":
        return False
    if "start" in f:
        return (date.fromisoformat(f["end"]) - date.fromisoformat(f["start"])).days > 300
    return True

kanonis = []  # (concept, value, unit, period_end, fy, fp, form, filed, accn)
ugaap = facts["facts"]["us-gaap"]
for concept in CONCEPTS:
    for f in ugaap[concept]["units"]["USD"]:
        if is_annual(f):
            kanonis.append((concept, f["val"], "USD", f["end"], f.get("fy"),
                            f.get("fp"), f["form"], f["filed"], f["accn"]))
print(f"tabel kanonis: {len(kanonis)} baris, kolom: concept|value|unit|end|fy|fp|form|filed|accn")
print("contoh:", kanonis[0][:4])

tabel kanonis: 168 baris, kolom: concept|value|unit|end|fy|fp|form|filed|accn
contoh: ('NetIncomeLoss', 17681000000, 'USD', '2008-06-30')


## Penjelasan (Indonesia)
Tahap **Parse** (FDP3): dari JSON bersarang menjadi tabel datar 9 kolom. Perhatikan fakta `instant` (Assets — neraca) tidak punya `start`, sedangkan `duration` (NetIncomeLoss — laba rugi) punya. Filter `is_annual` memisahkan fakta tahunan dari kuartalan — koneksi langsung ke M38 (flow vs stock).

In [1]:
# Tujuan cell ini: bersihkan — filed terbaru menang + provenance (tahap 3: Clean)
bersih = {}  # (concept, end) -> (value, filed, accn)
for concept, val, unit, end, fy, fp, form, filed, accn in kanonis:
    key = (concept, end)
    if key not in bersih or filed > bersih[key][1]:
        bersih[key] = (val, filed, accn)

restated = sum(1 for c in CONCEPTS for e in {k[1] for k in bersih if k[0] == c}
               if sum(1 for k in bersih if k == (c, e)) == 0)  # placeholder
# hitung periode dengan >1 filing (duplikat antar filing)
from collections import Counter
cnt = Counter((c, e) for c, _, _, e, _, _, _, _, _ in kanonis)
n_dup = sum(1 for k, v in cnt.items() if v > 1)
print(f"fakta unik (concept, end): {len(bersih)}; periode dengan duplikat antar-filing: {n_dup}")
latest_end = max(e for c, e in bersih if c == "NetIncomeLoss")
print("periode NI terbaru:", latest_end)

fakta unik (concept, end): 73; periode dengan duplikat antar-filing: 65
periode NI terbaru: 2026-06-30


## Penjelasan (Indonesia)
Tahap **Clean** (FDP4): aturan `filed` terbaru menang menyelesaikan duplikat antar-filing. Periode dengan >1 filing adalah kandidat restatement — pipeline menandai, bukan menyembunyikan. Provenance (`accn`) menempel di tiap angka.

In [1]:
# Tujuan cell ini: hitung rasio FY terbaru (tahap 4: Ratios)
E = latest_end
ni  = Decimal(bersih[("NetIncomeLoss", E)][0])
ta  = Decimal(bersih[("Assets", E)][0])
tl  = Decimal(bersih[("Liabilities", E)][0])
eq  = Decimal(bersih[("StockholdersEquity", E)][0])

rasio = {
    "net_margin_proxy": None,  # butuh revenue; NI/Assets sebagai gantinya:
    "roa": ni / ta * 100,
    "roe": ni / eq * 100,
    "debt_to_equity": tl / eq,
}
for k, v in rasio.items():
    if v is not None:
        print(f"{k} = {v:.2f}")
print(f"\nperiode: FY berakhir {E} | provenance NI: {bersih[('NetIncomeLoss', E)][2]}")

roa = 17.64
roe = 30.23
debt_to_equity = 0.71

periode: FY berakhir 2026-06-30 | provenance NI: 0001193125-26-323660


## Penjelasan (Indonesia)
Tahap **Ratios** (FDP5): tiga rasio dari angka yang sudah bersih. `net_margin` tidak dihitung karena concept revenue MSFT jarang di snapshot ini — pipeline yang jujur melewatkan rasio yang inputnya tak tersedia (skenario 4 peta fallback), bukan mengarang. Dipakai `Decimal` agar presisi uang terjaga (pelajaran FDP4).

In [1]:
# Tujuan cell ini: tulis dataset_kanonis.csv + self-check konsistensi (gerbang mini C3)
with open("dataset_kanonis.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["concept", "value", "unit", "period_end", "fy", "form", "filed", "accn"])
    for (concept, end), (val, filed, accn) in sorted(bersih.items()):
        fy = next(x[4] for x in kanonis if x[0] == concept and x[3] == end)
        form = next(x[6] for x in kanonis if x[0] == concept and x[3] == end)
        w.writerow([concept, val, "USD", end, fy, form, filed, accn])
print("dataset_kanonis.csv ditulis:", len(bersih), "baris")

# Self-check C3: Assets = Liabilities + Equity (harus balance eksak)
selisih = ta - (tl + eq)
print(f"\nC3 neraca balance: Assets − (Liab + Equity) = {selisih:,} ->",
      "LULUS ✓" if selisih == 0 else "GAGAL ❌")

dataset_kanonis.csv ditulis: 73 baris

C3 neraca balance: Assets − (Liab + Equity) = 0 -> LULUS ✓


## Penjelasan (Indonesia)
**Gerbang mini (C3):** sebelum angka diinterpretasikan, neraca harus balance eksak. Di sini selisihnya nol — lolos. Inilah "kepercayaan" yang dimaksud FDP6: bukan klaim "pipeline jalan", melainkan bukti "angka konsisten".

**Deliverable capstone:** (1) pipeline ini (jadikan `pipeline.py`), (2) `dataset_kanonis.csv` yang baru ditulis, (3) untuk Bagian B: jalankan `reconcile.py --benchmark benchmark.csv --input <mapping-idx-kamu.csv>` di folder `fdp-tooling/` hingga berstatus **LULUS**. Tiga file itu = definisi "selesai" track ini.